# 128-atom HCP noncollinear QE input from the last dataset step

This notebook creates a noncollinear Quantum ESPRESSO MD input for the 128-atom HCP Fe point closest to $c/a=1.60$.

The default dataset is `a_2.14_c_3.42_5000K.npz`, for which $c/a=1.59813$. The notebook uses the literal final position record (MD iteration 400), generates a zero-net noncollinear magnetic SQS, computes Maxwell–Boltzmann velocities at 5000 K, removes center-of-mass drift, rescales the formatted velocities to the exact target temperature, and verifies every generated QE card.

The final position record has no matching force block in the archive. This does not affect input creation because only its positions and fixed cell are reused; the velocities are generated afresh.

In [ ]:
from pathlib import Path
from pprint import pprint
import importlib
import json
import sys

import numpy as np

CWD = Path.cwd().resolve()
WORKSPACE_ROOT = None
for candidate in (CWD, *CWD.parents):
    if (candidate / "IronCoreMD" / "codes" / "prepare_latest_bcc_qe_input.py").exists():
        WORKSPACE_ROOT = candidate
        break
if WORKSPACE_ROOT is None:
    raise FileNotFoundError("Could not locate the Fe workspace root.")

REPO_ROOT = WORKSPACE_ROOT / "IronCoreMD"
CODES_DIR = REPO_ROOT / "codes"
if str(CODES_DIR) in sys.path:
    sys.path.remove(str(CODES_DIR))
sys.path.insert(0, str(CODES_DIR))

import prepare_latest_bcc_qe_input as qe_input_tools
import generate_qe_maxwell_velocities as qe_velocity
qe_input_tools = importlib.reload(qe_input_tools)
qe_velocity = importlib.reload(qe_velocity)

DATASET_DIR = WORKSPACE_ROOT / "dataset" / "hcp"
NPZ_PATH = DATASET_DIR / "a_2.14_c_3.42_5000K.npz"
OUTPUT_DIR = WORKSPACE_ROOT / "prepared_qe_inputs" / "hcp_noncollinear_a_2.14_c_3.42_5000K_last_frame"

FRAME_INDEX = -1  # literal final position record
TARGET_TEMPERATURE_K = 5000.0
VELOCITY_SEED = 5000128
SPIN_SEED = 20260730
STARTING_MAGNETIZATION = 0.35

PSEUDO_DIR = WORKSPACE_ROOT
PSEUDO_FILE = "Fe.pbe-spn-kjpaw_psl.1.0.0.UPF"
DT_AU = 20.670
NSTEP = 400
ECUTWFC = 71.0
ECUTRHO = 496.0
DEGAUSS = 0.02
K_GRID = (1, 1, 1, 0, 0, 0)
MIXING_BETA = 0.01
CONSTRAINED_MAGNETIZATION = True
LAMBDA_VALUE = 0.2

print(f"Dataset: {NPZ_PATH}")
print(f"Output:  {OUTPUT_DIR}")

In [ ]:
# Read the literal last valid position record and the corresponding fixed HCP cell.
with np.load(NPZ_PATH, allow_pickle=False) as data:
    positions_all = np.asarray(data["positions"], dtype=float)
    nframes, natoms, _ = positions_all.shape
    resolved_index = FRAME_INDEX if FRAME_INDEX >= 0 else nframes + FRAME_INDEX
    if resolved_index < 0 or resolved_index >= nframes:
        raise IndexError(f"Frame index {FRAME_INDEX} is out of range")

    cell_ang = qe_input_tools.frame_cell_angstrom(
        data, resolved_index, qe_input_tools.fixed_cell_angstrom(data)
    )
    frac_positions = qe_input_tools.wrap_fractional(
        qe_input_tools.frame_positions_fractional(data, resolved_index, cell_ang)
    )
    ideal_frac_positions = qe_input_tools.wrap_fractional(
        np.asarray(data["initial_positions_alat"], dtype=float)
        @ np.linalg.inv(np.asarray(data["initial_cell_alat"], dtype=float))
    )
    iteration = int(data["iteration"][resolved_index])
    time_ps = float(data["time_ps"][resolved_index])
    instantaneous_temperature_k = float(data["temperature_K"][resolved_index])
    position_complete = bool(np.isfinite(positions_all[resolved_index]).all())
    force_complete = bool(np.isfinite(data["forces_ry_au"][resolved_index]).all())

assert natoms == 128, f"Expected 128 HCP atoms, found {natoms}"
assert position_complete, "The selected HCP position frame is incomplete"
c_over_a = float(np.linalg.norm(cell_ang[2]) / np.linalg.norm(cell_ang[0]))

print(f"Frames in archive:       {nframes}")
print(f"Atoms per frame:         {natoms}")
print(f"Selected frame index:    {resolved_index}")
print(f"Selected MD iteration:   {iteration}")
print(f"Selected time:           {time_ps:.6f} ps")
print(f"Position record valid:   {position_complete}")
print(f"Matching force complete: {force_complete}")
print(f"Instantaneous dataset T: {instantaneous_temperature_k:.3f} K")
print(f"Velocity target T:       {TARGET_TEMPERATURE_K:.1f} K")
print(f"HCP c/a ratio:           {c_over_a:.8f}")

In [ ]:
# Generate the 128 atom-resolved noncollinear directions on the ideal HCP sites.
spins_cart, angle1, angle2, net_magnetization = qe_input_tools.generate_paramagnetic_spins(
    natoms,
    m_abs=STARTING_MAGNETIZATION,
    seed=SPIN_SEED,
    mode=qe_input_tools.SPIN_MODE_MAGNETIC_SQS,
    fractional_positions=ideal_frac_positions,
    cell_ang=cell_ang,
)
starting_magnetization = np.full(natoms, STARTING_MAGNETIZATION, dtype=float)
shells = qe_input_tools.magnetic_neighbor_shells(ideal_frac_positions, cell_ang)
shell_correlations = qe_input_tools.magnetic_shell_correlations(
    spins_cart / STARTING_MAGNETIZATION, shells
)

np.testing.assert_allclose(net_magnetization, 0.0, atol=1.0e-12, rtol=0.0)
print(f"Net starting magnetization vector: {net_magnetization}")
print("HCP magnetic-SQS shell correlations:")
print(" ".join(f"shell{i}={value:+.8f}" for i, value in enumerate(shell_correlations, start=1)))

In [ ]:
# Create the HCP QE base input, compute velocities, and insert the velocity card.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
prefix = "Fe_hcp_4x4x4_a2p14_c3p42_noncollinear_paramagnetic"
base_input_path = OUTPUT_DIR / f"{prefix}_5000K_base.in"
final_input_path = OUTPUT_DIR / f"{prefix}_5000K_with_velocities.in"
velocity_block_path = OUTPUT_DIR / f"{prefix}_atomic_velocities_5000K.txt"
spin_vectors_path = OUTPUT_DIR / f"{prefix}_spin_vectors.txt"
spin_parameters_path = OUTPUT_DIR / f"{prefix}_qe_spin_parameters.txt"

species_labels = qe_input_tools.generate_unique_qe_species_labels(natoms)
base_text = qe_input_tools.build_noncollinear_md_input(
    frac_positions,
    cell_ang,
    species_labels=species_labels,
    prefix=prefix,
    pseudo_dir=str(PSEUDO_DIR),
    pseudo_file=PSEUDO_FILE,
    outdir="./md",
    restart_mode="from_scratch",
    startingpot=None,
    startingwfc=None,
    temperature_k=TARGET_TEMPERATURE_K,
    dt_au=DT_AU,
    nstep=NSTEP,
    ecutwfc=ECUTWFC,
    ecutrho=ECUTRHO,
    degauss=DEGAUSS,
    k_grid=K_GRID,
    angle1=angle1,
    angle2=angle2,
    starting_magnetization=starting_magnetization,
    constrained_magnetization=CONSTRAINED_MAGNETIZATION,
    lambda_value=LAMBDA_VALUE,
    mixing_beta=MIXING_BETA,
    mixing_mode=None,
    mixing_ndim=None,
    diagonalization=None,
    nosym=True,
)
base_input_path.write_text(base_text)

labels, masses_amu = qe_velocity.read_qe_species_sequence(base_input_path)
sampled_velocities = qe_velocity.sample_maxwell_velocities_au(
    masses_amu, TARGET_TEMPERATURE_K, VELOCITY_SEED
)
velocities_au, velocity_block, measured_temperature_k = qe_velocity.finalize_velocities_for_qe(
    labels, sampled_velocities, masses_amu, TARGET_TEMPERATURE_K
)
final_text = qe_velocity.replace_or_append_atomic_velocities(base_text, velocity_block)

velocity_block_path.write_text(velocity_block)
final_input_path.write_text(final_text)
qe_input_tools.write_spin_vectors_txt(spins_cart, spin_vectors_path)
qe_input_tools.write_qe_spin_parameters_txt(
    angle1, angle2, spin_parameters_path, starting_magnetization=starting_magnetization
)

print(f"Base QE input:  {base_input_path}")
print(f"Final QE input: {final_input_path}")
print(f"Velocity card:  {velocity_block_path}")

In [ ]:
# Verify positions, cell, labels, COM drift, and the formatted velocity temperature.
def read_qe_positions(path: Path, count: int) -> tuple[list[str], np.ndarray]:
    lines = path.read_text().splitlines()
    start = next(i for i, line in enumerate(lines) if line.strip().upper().startswith("ATOMIC_POSITIONS"))
    rows = [lines[start + 1 + offset].split() for offset in range(count)]
    return [row[0] for row in rows], np.asarray([[float(x) for x in row[1:4]] for row in rows])

def read_qe_cell(path: Path) -> np.ndarray:
    lines = path.read_text().splitlines()
    start = next(i for i, line in enumerate(lines) if line.strip().upper().startswith("CELL_PARAMETERS"))
    return np.asarray([[float(x) for x in lines[start + offset].split()[:3]] for offset in (1, 2, 3)])

position_labels, written_positions = read_qe_positions(final_input_path, natoms)
written_cell = read_qe_cell(final_input_path)
velocity_labels, written_velocities = qe_velocity.parse_atomic_velocities(final_text.splitlines())
written_temperature_k = qe_velocity.temperature_from_velocities_au(
    written_velocities, masses_amu, remove_com=True
)
mass_weighted_velocity_sum = np.sum(masses_amu[:, None] * written_velocities, axis=0)

qe_input_tools.validate_label_consistency(species_labels, position_labels, velocity_labels)
np.testing.assert_allclose(written_positions, frac_positions, atol=5.1e-11, rtol=0.0)
np.testing.assert_allclose(written_cell, cell_ang, atol=5.1e-11, rtol=0.0)
np.testing.assert_allclose(mass_weighted_velocity_sum, 0.0, atol=1.0e-12, rtol=0.0)
np.testing.assert_allclose(written_temperature_k, TARGET_TEMPERATURE_K, atol=1.0e-8, rtol=0.0)

print("PASS: 128 final-step positions match the HCP dataset")
print("PASS: HCP cell matches the selected dataset point")
print("PASS: position and velocity labels match line-by-line")
print(f"PASS: mass-weighted COM velocity sum = {mass_weighted_velocity_sum}")
print(f"PASS: formatted velocity temperature = {written_temperature_k:.9f} K")

In [ ]:
# Write provenance and preview the final cards.
summary = {
    "phase": "hcp",
    "supercell": "4x4x4",
    "natoms": natoms,
    "ntypes": len(species_labels),
    "dataset": str(NPZ_PATH),
    "frame_index": resolved_index,
    "iteration": iteration,
    "time_ps": time_ps,
    "position_complete": position_complete,
    "matching_force_complete": force_complete,
    "a_angstrom": float(np.linalg.norm(cell_ang[0]) / 4.0),
    "c_angstrom": float(np.linalg.norm(cell_ang[2]) / 4.0),
    "c_over_a": c_over_a,
    "dataset_instantaneous_temperature_k": instantaneous_temperature_k,
    "target_velocity_temperature_k": TARGET_TEMPERATURE_K,
    "measured_velocity_temperature_k": float(written_temperature_k),
    "velocity_seed": VELOCITY_SEED,
    "spin_seed": SPIN_SEED,
    "spin_mode": "magnetic_sqs",
    "net_starting_magnetization": net_magnetization.tolist(),
    "magnetic_shell_correlations": shell_correlations.tolist(),
    "qe_input_base": str(base_input_path),
    "qe_input_final": str(final_input_path),
    "velocity_block": str(velocity_block_path),
    "spin_vectors": str(spin_vectors_path),
    "spin_parameters": str(spin_parameters_path),
    "position_velocity_labels_match": position_labels == velocity_labels,
}
summary_path = OUTPUT_DIR / "generation_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")
pprint(summary)

print("\nATOMIC_POSITIONS preview:")
for label, position in list(zip(position_labels, written_positions))[:5]:
    print(f"{label:4s} {position[0]: .10f} {position[1]: .10f} {position[2]: .10f}")

print("\nATOMIC_VELOCITIES preview:")
print("\n".join(velocity_block.splitlines()[:6]))
print(f"\nSummary: {summary_path}")